# GNN with Data Leakage Fixed
**Fix**: edge train/test split is performed *before* any node feature computation.
All graph-based features (degree, clustering, betweenness, eigenvector centrality) are computed on the training graph only.

In [ ]:
!pip install torch torch_geometric scikit-learn pandas numpy matplotlib seaborn networkx

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.utils import negative_sampling
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, roc_auc_score, average_precision_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import networkx as nx
import json
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load data
with pd.HDFStore('./data/gene_network_data.h5') as store:
    tec = store['TEC']

np_tec_abs = np.abs(tec.to_numpy(copy=True))
gene_names = tec.index.tolist()
n_genes = len(gene_names)
print(f'Loaded {n_genes} genes x {tec.shape[1]} cell types')

In [ ]:
# ── STEP 1: Edge split BEFORE any feature computation (leakage fix) ──────────
np.random.seed(42)
threshold = 0.75

adj_matrix = (np_tec_abs > threshold).astype(float)
np.fill_diagonal(adj_matrix, 0)

# All positive edges (upper triangle)
pos_rows, pos_cols = np.where(np.triu(adj_matrix, k=1) > 0)
all_pos_edges = np.vstack([pos_rows, pos_cols])
n_pos = all_pos_edges.shape[1]

# 80/20 split
perm = np.random.permutation(n_pos)
split = int(0.8 * n_pos)
train_edges    = all_pos_edges[:, perm[:split]]
test_pos_edges = all_pos_edges[:, perm[split:]]

# Training adjacency matrix (training edges only)
adj_train = np.zeros((n_genes, n_genes), dtype=float)
adj_train[train_edges[0], train_edges[1]] = 1.0
adj_train[train_edges[1], train_edges[0]] = 1.0

# Bidirectional train edge_index for PyG
train_edge_index = torch.tensor(
    np.hstack([train_edges, train_edges[[1, 0], :]]),
    dtype=torch.long
).to(device)

# Test negative edges (sampled from non-edges of full graph)
neg_rows, neg_cols = np.where(np.triu(adj_matrix == 0, k=1))
neg_perm = np.random.choice(len(neg_rows), size=test_pos_edges.shape[1], replace=False)
test_neg_edges = np.vstack([neg_rows[neg_perm], neg_cols[neg_perm]])

print(f'Total positive edges: {n_pos}')
print(f'Train: {train_edges.shape[1]} | Test pos: {test_pos_edges.shape[1]} | Test neg: {test_neg_edges.shape[1]}')

In [ ]:
# ── STEP 2: Node features computed on TRAINING graph only ────────────────────

# Degree from training adjacency
degrees = np.sum(adj_train, axis=1)

# Clustering coefficient and centralities via NetworkX on training graph
G_train = nx.from_numpy_array(adj_train)

clust_dict = nx.clustering(G_train)
clustering = np.array([clust_dict[i] for i in range(n_genes)])
print('Clustering done.')

betw_dict = nx.betweenness_centrality(G_train, k=300, normalized=True, seed=42)
betweenness = np.array([betw_dict[i] for i in range(n_genes)])
print('Betweenness done.')

try:
    eig_dict = nx.eigenvector_centrality_numpy(G_train)
    eigenvector = np.array([eig_dict[i] for i in range(n_genes)])
except Exception:
    eigenvector = degrees / (degrees.max() + 1e-8)
print('Eigenvector done.')

# Correlation moments (marginal stats — no graph topology, no leakage)
mean_corr = np.mean(np_tec_abs, axis=1)
std_corr  = np.std(np_tec_abs, axis=1)
max_corr  = np.max(np_tec_abs, axis=1)

# Stack and normalise
node_features = np.column_stack([degrees, clustering, mean_corr, std_corr, max_corr, betweenness, eigenvector])
node_features = (node_features - node_features.mean(axis=0)) / (node_features.std(axis=0) + 1e-8)
x = torch.tensor(node_features, dtype=torch.float32).to(device)
print(f'Node features shape: {x.shape}  (computed on training graph only)')

In [ ]:
# Model
class TEC_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, out_dim=64, num_heads=8, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(input_dim, hidden_dim, heads=num_heads, dropout=dropout, concat=True)
        self.bn1  = nn.BatchNorm1d(hidden_dim * num_heads)
        self.gat2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=4, dropout=dropout, concat=True)
        self.bn2  = nn.BatchNorm1d(hidden_dim * 4)
        self.gat3 = GATConv(hidden_dim * 4, out_dim, heads=1, dropout=dropout, concat=False)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn1(self.gat1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.gat2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat3(x, edge_index)
        return F.normalize(x, p=2, dim=1)


def link_prediction_loss(embeddings, pos_edge_index, neg_edge_index):
    pos_scores = (embeddings[pos_edge_index[0]] * embeddings[pos_edge_index[1]]).sum(dim=1)
    neg_scores = (embeddings[neg_edge_index[0]] * embeddings[neg_edge_index[1]]).sum(dim=1)
    pos_loss = -F.logsigmoid(pos_scores).mean()
    neg_loss = -F.logsigmoid(-neg_scores).mean()
    mean_emb = embeddings.mean(dim=0)
    diversity_loss = F.mse_loss(mean_emb, torch.zeros_like(mean_emb))
    return pos_loss + neg_loss - 0.1 * diversity_loss


print('Model defined.')

In [ ]:
# Training
model = TEC_GNN(input_dim=x.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

train_edge_tensor = torch.tensor(train_edges, dtype=torch.long).to(device)

best_loss = float('inf')
patience_counter = 0
start_time = time.time()
losses = []

model.train()
for epoch in range(300):
    optimizer.zero_grad()
    embeddings = model(x, train_edge_index)

    neg_edge_index = negative_sampling(
        edge_index=train_edge_index,
        num_nodes=n_genes,
        num_neg_samples=train_edges.shape[1]
    )

    loss = link_prediction_loss(embeddings, train_edge_tensor, neg_edge_index)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step(loss)
    losses.append(loss.item())

    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_noleak.pt')
    else:
        patience_counter += 1

    if patience_counter >= 20:
        print(f'Early stopping at epoch {epoch}')
        break

    if epoch % 50 == 0:
        print(f'Epoch {epoch}: Loss={loss.item():.4f}, LR={optimizer.param_groups[0]["lr"]:.6f}')

training_time = time.time() - start_time
print(f'Training complete in {training_time:.1f}s, best loss={best_loss:.4f}')

model.load_state_dict(torch.load('best_model_noleak.pt'))
model.eval()
with torch.no_grad():
    embeddings_np = model(x, train_edge_index).cpu().numpy()

print(f'Embedding std: {embeddings_np.std():.4f}')

In [ ]:
# Collapse check
sim_matrix = cosine_similarity(embeddings_np)
np.fill_diagonal(sim_matrix, 0)
upper_tri = sim_matrix[np.triu_indices(n_genes, k=1)]

print('Similarity stats (should NOT all be ~1.0):')
print(f'  Mean: {upper_tri.mean():.4f}')
print(f'  Std:  {upper_tri.std():.4f}')
print(f'  % > 0.99: {(upper_tri > 0.99).mean()*100:.1f}%  (should be << 100%)')

In [ ]:
# Evaluation on held-out test edges
pos_scores = (embeddings_np[test_pos_edges[0]] * embeddings_np[test_pos_edges[1]]).sum(axis=1)
neg_scores = (embeddings_np[test_neg_edges[0]] * embeddings_np[test_neg_edges[1]]).sum(axis=1)

all_scores = np.concatenate([pos_scores, neg_scores])
all_labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])

auc_roc = roc_auc_score(all_labels, all_scores)
auc_pr  = average_precision_score(all_labels, all_scores)

print(f'\n=== HELD-OUT TEST RESULTS (no data leakage) ===')
print(f'AUC-ROC: {auc_roc:.4f}')
print(f'AUC-PR:  {auc_pr:.4f}')

In [ ]:
# Clustering
best_k, best_sil = 2, -1
cluster_labels = None
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_np)
    s = silhouette_score(embeddings_np, labels)
    print(f'k={k}: silhouette={s:.4f}')
    if s > best_sil:
        best_sil = s
        best_k = k
        cluster_labels = labels

print(f'\nBest k={best_k}, silhouette={best_sil:.4f}')
print(f'Cluster sizes: {np.bincount(cluster_labels)}')

In [ ]:
# Plot
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings_np)
for c in range(best_k):
    mask = cluster_labels == c
    axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1], s=1, alpha=0.3, label=f'Cluster {c} (n={mask.sum()})')
axes[0].set_title(f'GNN Embeddings (PCA)\nSilhouette={best_sil:.3f}', fontsize=12)
axes[0].legend(markerscale=5)

axes[1].hist(upper_tri, bins=50, color='#2196F3', alpha=0.7)
axes[1].set_xlabel('Cosine Similarity', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title(f'Similarity Distribution\nMean={upper_tri.mean():.3f}, Std={upper_tri.std():.3f}', fontsize=12)

fpr, tpr, _ = roc_curve(all_labels, all_scores)
axes[2].plot(fpr, tpr, color='#2196F3', lw=2, label=f'GNN (AUC={auc_roc:.3f})')
axes[2].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
axes[2].set_xlabel('False Positive Rate', fontsize=12)
axes[2].set_ylabel('True Positive Rate', fontsize=12)
axes[2].set_title('ROC Curve (Held-out Test Set)', fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.savefig('noleak_gnn_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: noleak_gnn_results.png')

In [ ]:
# Save results
results = {
    'leakage_fixed': True,
    'features_computed_on': 'training_graph_only',
    'training_time': training_time,
    'best_loss': best_loss,
    'auc_roc': auc_roc,
    'auc_pr': auc_pr,
    'best_k': best_k,
    'silhouette': best_sil,
    'cluster_sizes': np.bincount(cluster_labels).tolist(),
    'embedding_mean_similarity': float(upper_tri.mean()),
    'embedding_std_similarity': float(upper_tri.std()),
    'pct_similarity_above_099': float((upper_tri > 0.99).mean() * 100)
}
with open('noleak_gnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

np.save('noleak_gnn_embeddings.npy', embeddings_np)
np.save('noleak_gnn_cluster_labels.npy', cluster_labels)
pd.DataFrame({'gene': gene_names, 'cluster': cluster_labels}).to_csv(
    'noleak_gene_cluster_assignments.csv', index=False
)

print('\n=== FINAL SUMMARY ===')
print(f'AUC-ROC:    {auc_roc:.4f}')
print(f'AUC-PR:     {auc_pr:.4f}')
print(f'Silhouette: {best_sil:.4f}')
print(f'Clusters:   {best_k}')
print('All results saved.')